In [ ]:



import pandas as pd
import sqlite3
import os
import time
import gc
import logging
logging.basicConfig(level=logging.INFO,
                    filename='logs.log',
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    filemode='a')
print("Ready ")

Ready ✓


In the above cell we imported modules like 
1) OS : allows python to talk our system
2)Pandas: opensource library designed for data manipulation and analysis.
3)Logging: To create log file which saves the basic configuration like data ingesting time for data to import in database.
4)GC :It is garbage collector.

In [ ]:

# Here we use sqlite3 to load the data directly into a SQLite database. 

data_path="C:\\Vendor project\\Data"
def load_data_fast(data_path="C:\\Vendor project\\Data", db_path="data1.db"):
    
    # Direct SQLite 
    conn = sqlite3.connect(db_path)
    
    files = [f for f in os.listdir(data_path) if f.endswith(".csv")]
    print(f"Found {len(files)} CSV files\n")

    for file in files:
        full_path = os.path.join(data_path, file)
        table_name = file[:-4].replace(" ", "_")
        file_size = os.path.getsize(full_path) / (1024**2) 
        print(f" {file} ({file_size:.1f} MB)")

        start = time.time()
        first_chunk = True
        rows_loaded = 0

        try:
            for chunk in pd.read_csv(
                full_path,
                chunksize=5000,         
                low_memory=True,
                on_bad_lines='skip',
                encoding='utf-8',
                encoding_errors='replace'
            ):
                chunk.to_sql(
                    table_name,
                    conn,
                    if_exists='replace' if first_chunk else 'append',
                    index=False,
                    method='multi'      
                )
                rows_loaded += len(chunk)
                first_chunk = False
                del chunk
                gc.collect()
                print(f"   {rows_loaded:,} rows...", end="\r")

            elapsed = time.time() - start
            print(f"  {rows_loaded:,} rows in {elapsed:.1f}s  ")

        except Exception as e:
            print(f"    Failed: {e}")

    conn.commit()
    conn.close()
    print("\n All files loaded!")

print("Function ready ✓")

Function ready ✓


In [24]:
load_data_fast(data_path)

Found 0 Excel files


🎉 All files loaded!
